# Links

- [Liu 2025: Cell type–specific 3D-genome organization and transcription regulation in the brain](https://www.science.org/doi/10.1126/sciadv.adv2067)
- [GitHub repo for Liu 2025](https://github.com/ZhuangLab/Chromatin_Analysis_MOp/)

# Processing FOF-CT files

## Docs

- [Docs: 4DN FISH Omics Format - Chromatin Tracing (FOF-CT)](https://fish-omics-format.readthedocs.io/en/latest/)
- [DNA-Spot/Trace Data core table](https://fish-omics-format.readthedocs.io/en/latest/core.html)
- [Cell Data table](https://fish-omics-format.readthedocs.io/en/latest/cell.html)

Note: FOV = 'fields of view'

## Examining files

### Examining DNA-Spot/Trace Data core table

In [16]:
%%bash

sample_dir="$HOME/noble_lab/repos/Chromatin_Analysis_MOp/data/MOp_wt"
cd "$sample_dir/biorep1_techrep1"

# # ==== Examine header info

# Number of header lines
echo $(grep '^["#]' dna_spot_trace_core.csv | wc -l)" header lines, including col labels"
echo $(grep ',,,' dna_spot_trace_core.csv | wc -l)" header lines, excluding col labels"

# Column labels
echo -e "\nColumn labels:"
grep -v ',,,' dna_spot_trace_core.csv | head -n 1 | sed 's/.*(//;s/).*//' | tr ',' '\n' | awk 'BEGIN{c=1}{print c "\t" $0; c += 1}'

# Column label desc
echo -e "\nColumn label desc:"
grep ',,,' dna_spot_trace_core.csv | tail -n 7 | sed 's/,,*//g'

23 header lines, including col labels
22 header lines, excluding col labels

Column labels:
1	Spot_ID
2	Trace_ID
3	X
4	Y
5	Z
6	Chrom
7	Chrom_Start
8	Chrom_End
9	Chrom_order
10	Cell_ID
11	FOV_ID
12	CellID_byFOV
13	RNA_experiment_ID
14	DNA_experiment_ID
15	Sample_ID

Column label desc:
##XYZ_unit=nm
#Spot_ID:unique DNA spot identifier across 4dn_FOF-CT_core and 4dn_FOF-CT_demultiplxeing for the same replicate
#Trace_ID:number format as fovXX_cellXX_chrXX_chromatidXX
#Chrom_order:order of DNA region on its respective chromosome
#Cell_ID:unique cell identifier across the datatables
#FOV:FOV ID across datatables for the same replicate
#CellID_byFOVspecific cell identifier relative to the FOV


In [9]:
%%bash

sample_dir="$HOME/noble_lab/repos/Chromatin_Analysis_MOp/data/MOp_wt"
cd "$sample_dir/biorep1_techrep1"

# ==== Check uniqueness of cells
echo -e "Uniqueness of cells"
echo $(grep -v '^["#]' dna_spot_trace_core.csv | cut -d ',' -f 10-12 | sort | uniq | wc -l)" unique combos of 'Cell_ID, FOV_ID, CellID_byFOV'"
echo $(grep -v '^["#]' dna_spot_trace_core.csv | cut -d ',' -f 10 | sort | uniq | wc -l)" unique 'Cell_ID'"
echo $(grep -v '^["#]' dna_spot_trace_core.csv | cut -d ',' -f 11-12 | sort | uniq | wc -l)" unique combos of 'FOV_ID, CellID_byFOV'"
echo $(grep -v '^["#]' dna_spot_trace_core.csv | cut -d ',' -f 12 | sort | uniq | wc -l)" unique 'CellID_byFOV'"
echo $(grep -v '^["#]' dna_spot_trace_core.csv | cut -d ',' -f 11 | sort | uniq | wc -l)" unique 'FOV_ID'"


# ==== Check uniqueness of rows
echo -e "\nUniqueness of rows"
echo $(grep -v '^["#]' dna_spot_trace_core.csv | wc -l)" unique 'rows'"
echo $(grep -v '^["#]' dna_spot_trace_core.csv | cut -d ',' -f 1 | sort | uniq | wc -l)" unique 'Spot_ID'"
echo $(grep -v '^["#]' dna_spot_trace_core.csv | cut -d ',' -f 2,9 | sort | uniq | wc -l)" unique combos of 'Trace_ID, Chrom_order'"


Uniqueness of cells
12725 unique combos of 'Cell_ID, FOV_ID, CellID_byFOV'
12725 unique 'Cell_ID'
12725 unique combos of 'FOV_ID, CellID_byFOV'
253 unique 'CellID_byFOV'
148 unique 'FOV_ID'

Uniqueness of rows
8258329 unique 'rows'
8258329 unique 'Spot_ID'
8258329 unique combos of 'Trace_ID, Chrom_order'


In [11]:
%%bash

sample_dir="$HOME/noble_lab/repos/Chromatin_Analysis_MOp/data/MOp_wt"
cd "$sample_dir/biorep1_techrep1"

# ==== Check that the chromosome component of 'Trace_ID' (fovXX_cellXX_chrXX_chromatidXX) corresponds to the 'Chrom' column
ndiff=$(awk -F '[,_]' '{ if ($1 !~ /^[#"]/){ gsub(/chr/, "", $9); if ($4 != $9) { print $4 "\t" $9 } } }' dna_spot_trace_core.csv | wc -l)
if [[ $ndiff -eq 0 ]]; then
    echo "✓ Chromosome component of 'Trace_ID' (fovXX_cellXX_chrXX_chromatidXX) corresponds to the 'Chrom' column"
else
    echo "✗ Chromosome component of 'Trace_ID' (fovXX_cellXX_chrXX_chromatidXX) does NOT correspond to the 'Chrom' column for $ndiff lines"
fi


# ==== Examine size of each region
echo -e "\nSize of each region: (1st 10 lines)"
awk -F ',' '{ if ($1 !~ /^[#"]/){ print $8 - $7 } }' dna_spot_trace_core.csv | head

✓ Chromosome component of 'Trace_ID' (fovXX_cellXX_chrXX_chromatidXX) corresponds to the 'Chrom' column

Size of each region: (1st 10 lines)
8595
8623
11242
10504
8112
9570
8405
7269
8405
7187


### Examining Cell Data table

In [55]:
%%bash

sample_dir="$HOME/noble_lab/repos/Chromatin_Analysis_MOp/data/MOp_wt"
cd "$sample_dir/biorep1_techrep1"

# # ==== Examine header info

# Number of header lines
echo $(grep '^["#]' cell_data.csv | wc -l)" header lines, including col labels"
echo $(grep ',,,' cell_data.csv | wc -l)" header lines, excluding col labels"

# Column labels
ncols_gene=$( grep -v ',,,' cell_data.csv | head -n 1 | sed 's/.*neuron_identity,//' | tr ',' '\n' | wc -l )
echo -e "\nColumn labels (excluding $ncols_gene gene xp columns):"
grep -v ',,,' cell_data.csv | head -n 1 | sed 's/neuron_identity,.*/neuron_identity/;s/.*(//;s/).*//' | tr ',' '\n' | awk 'BEGIN{c=1}{print c "\t" $0; c += 1}'

# Column label desc
echo -e "\nColumn label desc:"
grep ',,,' cell_data.csv | tail -n 7 | sed 's/,,*//g'

23 header lines, including col labels
22 header lines, excluding col labels

Column labels (excluding 242 gene xp columns):
1	Cell_ID
2	FOV_ID
3	cell_volume_from_merlin
4	cell_center_x_global
5	cell_center_y_global
6	RNA_experiment_ID
7	Sample_ID
8	cluster_subclass
9	cluster_class
10	neuron_identity

Column label desc:
##XYZ_unit=micron
#Cell_ID:unique cell identifier across the datatables
#cell_center_x_global:x coordinate of the cell relative to all cells for the experiment replicate
#cell_center_y_global:y coordinate of the cell relative to all cells for the experiment replicate
#cluster_subclass:transcriptionally defined cell cluster lable at subclass level
#cluster_class:transcriptionally defined cell cluster lable at a higer level
"#RNA count for gene A:decoded raw RNA count for gene A (e.g. gene 1700022I11Rik)"


In [116]:
%%bash

sample_dir="$HOME/noble_lab/repos/Chromatin_Analysis_MOp/data/MOp_wt"
biorep="$sample_dir/biorep1_techrep1"

# # ==== Count cells & clusters

# Number of cells
echo -e $(grep -v '^["#]' "$biorep/cell_data.csv" | wc -l)" cells (lines)"

# Number of clusters
nclust_sub=$( grep -v '^["#]' "$biorep/cell_data.csv" | cut -d ',' -f 8 | sort | sed 's/ //g' | uniq -c | awk '{ print $1 "\t" $2 }' | sort -k1,1nr )
echo -e "\n"$(echo -e "$nclust_sub" | wc -l)" clusters (only via 'cluster_subclass')"
nclust=$( grep -v '^["#]' "$biorep/cell_data.csv" | cut -d ',' -f 8-10 | sort | sed 's/ //g' | uniq -c | awk '{ print $1 "\t" $2 }' | sort -k1,1nr )
echo -e $(echo -e "$nclust" | wc -l)" clusters (via combo of 'cluster_subclass, cluster_class, neuron_identity'):"
echo -e "$nclust"

# Number of clusters (only via cluster_class)
nclust_class=$( grep -v '^["#]' "$biorep/cell_data.csv" | cut -d ',' -f 9 | sort | sed 's/ //g' | uniq -c | awk '{ print $1 "\t" $2 }' | sort -k1,1nr )
echo -e "\n"$(echo -e "$nclust_class" | wc -l)" clusters (only via 'cluster_class')"
echo -e "$nclust_class"

17856 cells (lines)

22 clusters (only via 'cluster_subclass')
22 clusters (via combo of 'cluster_subclass, cluster_class, neuron_identity'):
2184	L2/3IT,Gluta,Neuronal
1803	Astro,Astro,Non-Neuronal
1741	L6CT,Gluta,Neuronal
1474	L5IT,Gluta,Neuronal
1294	other,other,other
1088	L5ET,Gluta,Neuronal
1039	Endo,Endo,Non-Neuronal
1029	L4/5IT,Gluta,Neuronal
1029	OPC,Oligo,Non-Neuronal
972	Oligo,Oligo,Non-Neuronal
789	L6IT,Gluta,Neuronal
711	Micro,Micro,Non-Neuronal
549	Pvalb,GABA,Neuronal
391	L5/6NP,Gluta,Neuronal
380	Sst,GABA,Neuronal
274	VLMC,VLMC,Non-Neuronal
266	Peri,Peri,Non-Neuronal
205	Vip,GABA,Neuronal
198	SMC,SMC,Non-Neuronal
185	L6b,Gluta,Neuronal
180	Lamp5,GABA,Neuronal
75	Sncg,GABA,Neuronal

10 clusters (only via 'cluster_class')
8881	Gluta
2001	Oligo
1803	Astro
1389	GABA
1294	other
1039	Endo
711	Micro
274	VLMC
266	Peri
198	SMC


## Extracting relevant information

In [51]:
%%bash

sample_dir="$HOME/noble_lab/repos/Chromatin_Analysis_MOp/data/MOp_wt"

# ==== Reformat relevant columns from 'cell_data.csv':
# 1 	Cell_ID
# 8 	cluster_subclass
# 9 	cluster_class
# 10	neuron_identity

# # Testing on one biorep
# biorep="$sample_dir/biorep1_techrep1"
# rep_id=$(basename "$biorep" | sed 's/_/./;s/[A-z]*//g' )
# awk -F ',' -v rep_id="$rep_id" 'BEGIN{OFS=","}{ if ($(NF-1) != ""){ 
#     if ($1 ~ /Cell_ID/){ print "#rep_id", "#" tolower($10), "#" tolower($9), "#" tolower($8), "#cell_id" }
# 	  else { gsub(/ /, "_", $8); print rep_id, tolower($10), tolower($9), tolower($8), $1 } } }' "$biorep/cell_data.csv" | sort -t ',' -k2,5 | head  | tr ',' '\t'


for biorep in $( find "$sample_dir" -type d -name 'biorep*_techrep*' ); do
	if [ -f "$biorep/cells.csv" ]; then
		continue
	fi
    basename "$biorep"
	if [ ! -f "$biorep/cell_data.csv" ] & [ -f "$biorep/cell_data.csv.gz" ]; then
		gunzip "$biorep/cell_data.csv.gz"
	fi

    rep_id=$(basename "$biorep" | sed 's/_/./;s/[A-z]*//g' )
    awk -F ',' -v rep_id="$rep_id" 'BEGIN{OFS=","}{ if ($(NF-1) != ""){ 
        if ($1 ~ /Cell_ID/){ print "#rep_id", "#" tolower($10), "#" tolower($9), "#" tolower($8), "#cell_id" }
        else { gsub(/ /, "_", $8); gsub(/\//, "-", $8); print rep_id, tolower($10), tolower($9), tolower($8), $1 } } }' "$biorep/cell_data.csv" | sort -t ',' -k2,5 > "$biorep/cells.csv"

	gzip "$biorep/cell_data.csv"
done

biorep3_techrep1
biorep1_techrep1
biorep2_techrep1
biorep3_techrep2


In [222]:
%%bash

sample_dir="$HOME/noble_lab/repos/Chromatin_Analysis_MOp/data/MOp_wt"

# ==== Reformat relevant columns from 'dna_spot_trace_core.csv':
# 2 	Trace_ID  (fovXX_cellXX_chrXX_chromatidXX)
# 3 	X
# 4 	Y
# 5 	Z
# 7 	Chrom_Start
# 8 	Chrom_End
# 9 	Chrom_order
# 10	Cell_ID

# # Testing on one biorep
# biorep="$sample_dir/biorep1_techrep1"
# rep_id=$(basename "$biorep" | sed 's/_/./;s/[A-z]*//g' )
# awk -F ',' -v rep_id="$rep_id" 'BEGIN{OFS=","}{ if ($(NF-1) != ""){ 
# 	df_cell_desc=$9","$7","$8","$3","$4","$5;
# 	if ($2 == "Trace_ID"){ molecule="#rep_id,#cell_id,#chrom,#trace_id"; df_cell_desc="#"gensub(/,/, ",#", "g", tolower(df_cell_desc)) }
# 	else { molecule=rep_id","$10","gensub(/.+_.+_(.+)_(.+)/, "\\1,\\2", "g", $2) };
# 	print molecule, df_cell_desc } }' "$biorep/dna_spot_trace_core.csv" | sort -t ',' -k2,2 -k3,3n -k4,4n -k5,5n | head | tr ',' '\t' | sed -r 's/(\.[0-9])[0-9]*/\1/g'


for biorep in $( find "$sample_dir" -type d -name 'biorep*_techrep*' ); do
	if [ -f "$biorep/dna_merfish.csv" ]; then
		continue
	fi
    basename "$biorep"
	if [ ! -f "$biorep/dna_spot_trace_core.csv" ] & [ -f "$biorep/dna_spot_trace_core.csv.gz" ]; then
		gunzip "$biorep/dna_spot_trace_core.csv.gz"
	fi

    rep_id=$(basename "$biorep" | sed 's/_/./;s/[A-z]*//g' )
    awk -F ',' -v rep_id="$rep_id" 'BEGIN{OFS=","}{ if ($(NF-1) != ""){ 
    	df_cell_desc=$9","$7","$8","$3","$4","$5;
    	if ($2 == "Trace_ID"){ molecule="#rep_id,#cell_id,#chrom,#trace_id"; df_cell_desc="#"gensub(/,/, ",#", "g", tolower(df_cell_desc)) }
    	else { molecule=rep_id","$10","gensub(/.+_.+_(.+)_(.+)/, "\\1,\\2", "g", $2) };
    	print molecule, df_cell_desc } }' "$biorep/dna_spot_trace_core.csv" | sort -t ',' -k2,2 -k3,3n -k4,4n -k5,5n > "$biorep/dna_merfish.csv"

	gzip "$biorep/dna_spot_trace_core.csv"
done

## Consolidating cell data across all replicates

In [897]:
%%bash

sample_dir="$HOME/noble_lab/repos/Chromatin_Analysis_MOp/data/MOp_wt"

allreps_dir="$sample_dir/all_reps"
cells_file="$allreps_dir/cells.csv"

# ==== Consolidate cells across all replicates

mkdir -p "$allreps_dir"

# Write header
head -n 1 $(find "$sample_dir" -type f -path '*/biorep*_techrep*/cells.csv' | head -n 1) > "$cells_file"

# Write cells (but only if DNA-MERFISH data exists for that cell)
# grep -r "$sample_dir/"biorep*_techrep*/cells.csv -hve '^#' | sort -t ',' -k1,5 >> "$cells_file" # Write all cells - OLD
for biorep in $( find "$sample_dir" -type d -name 'biorep*_techrep*' ); do
    join -t ',' -1 5 -2 1 --header --check-order \
    <( sort -t ',' -k 5,5 "$biorep/cells.csv" ) \
    <( cut -d ',' -f 2 "$biorep/dna_merfish.csv" | sort | uniq ) \
    | awk -F',' 'BEGIN{OFS=","}{if ($1 !~ /^#/) {print $2, $3, $4, $5, $1}}' >> "$cells_file"
done

# Sort cells in 'all reps' file
sort -t ',' -k1,5 -o "$cells_file" "$cells_file"

# Total cells
echo "Total cells (across all experiments): "$( grep -r "$sample_dir/"biorep*_techrep*/cells.csv -hve '^#' | wc -l )
echo "Total cells (that are present in DNA-MERFISH experiments): "$( grep -v '^#' "$cells_file" | wc -l )

Total cells (across all experiments): 62732
Total cells (that are present in DNA-MERFISH experiments): 46295


In [899]:
%%bash

sample_dir="$HOME/noble_lab/repos/Chromatin_Analysis_MOp/data/MOp_wt"

allreps_dir="$sample_dir/all_reps"
cells_file="$allreps_dir/cells.csv"

# ==== Count cells & clusters across all replicates

# Number of cells
echo -e $(grep -v '^["#]' "$cells_file" | wc -l)" lines in "$(basename "$cells_file")
echo -e $(grep -v '^["#]' "$cells_file" | cut -d ',' -f5 | sort | uniq | wc -l)" unique cells in "$(basename "$cells_file")

# Number of clusters
nclust=$( grep -v '^["#]' "$cells_file" | cut -d ',' -f 4 | sort | sed 's/ //g' | uniq -c | awk '{ print $1 "\t" $2 }' | sort -k1,1nr )
echo -e "\n"$(echo -e "$nclust" | wc -l)" clusters (only via 'cluster_subclass')"
nclust_trio=$( grep -v '^["#]' "$cells_file" | cut -d ',' -f 2-4 | sort | sed 's/ //g' | uniq -c | awk '{ print $1 "\t" $2 }' | sort -k1,1nr )
echo -e $(echo -e "$nclust_trio" | wc -l)" clusters (via combo of 'cluster_subclass, cluster_class, neuron_identity'):"
echo -e "$nclust_trio"

echo -e "\nNumber of (subclass-based) clusters with >= [cutoff] cells per cluster (ignoring 'other' cluster)"
for cutoff in 1000 2000 2500; do
    echo -e "$nclust" | awk -vcutoff="$cutoff" 'BEGIN{ nlines=0; total=0 } \
        {if (($1 >= cutoff) && ($2 != "other")){nlines += 1; total += $1}} \
        END{ print nlines" clusters with >= "cutoff" cells per cluster, total cells="total }'
done

# Number of clusters (only via cluster_class)
nclust_class=$( grep -v '^["#]' "$cells_file" | cut -d ',' -f 3 | sort | sed 's/ //g' | uniq -c | awk '{ print $1 "\t" $2 }' | sort -k1,1nr )
echo -e "\n"$(echo -e "$nclust_class" | wc -l)" clusters (only via 'cluster_class')"
echo -e "$nclust_class"

46295 lines in cells.csv
46295 unique cells in cells.csv

22 clusters (only via 'cluster_subclass')
22 clusters (via combo of 'cluster_subclass, cluster_class, neuron_identity'):
6469	non-neuronal,oligo,oligo
5463	neuronal,gluta,l6_ct
4830	non-neuronal,astro,astro
4373	neuronal,gluta,l2-3_it
3687	non-neuronal,endo,endo
3370	neuronal,gluta,l4-5_it
2659	neuronal,gluta,l5_it
2510	neuronal,gluta,l6_it
1867	non-neuronal,oligo,opc
1856	non-neuronal,micro,micro
1637	neuronal,gluta,l5_et
1338	neuronal,gaba,pvalb
1036	neuronal,gluta,l6b
969	neuronal,gaba,sst
939	non-neuronal,peri,peri
751	neuronal,gluta,l5-6_np
600	non-neuronal,vlmc,vlmc
555	non-neuronal,smc,smc
465	neuronal,gaba,vip
455	neuronal,gaba,lamp5
294	other,other,other
172	neuronal,gaba,sncg

Number of (subclass-based) clusters with >= [cutoff] cells per cluster (ignoring 'other' cluster)
13 clusters with >= 1000 cells per cluster, total cells=41095
8 clusters with >= 2000 cells per cluster, total cells=33361
8 clusters with >= 2500 c

## Segregating DNA-MERFISH data per cluster (excluding 'other' cluster)

In [900]:
%%bash

sample_dir="$HOME/noble_lab/repos/Chromatin_Analysis_MOp/data/MOp_wt"
ncells_cutoff=2000

allreps_dir="$sample_dir/all_reps"
cells_file="$allreps_dir/cells.csv"

# ==== Get cells belonging to 'top' clusters (clusters with >= cutoff cells)
clusters_all=$( grep -v '^["#]' "$cells_file" | cut -d ',' -f 4 | sort | sed 's/ //g' | uniq -c | awk '{ print $1 "\t" $2 }' | sort -k1,1nr )
clusters_top=$( echo -e "$clusters_all" | awk -vcutoff="$ncells_cutoff" '{if (($1 >= cutoff) && ($2 != "other")) {print $2}}' )
nclust_top=$( echo -e "$clusters_top" | wc -l )

clusters_top_re="(^"$( echo -e "$clusters_top" | tr '\n' '|' | sed 's/|$//;s/|/$|^/g;s:/:.:g' )"$)"
cells_topclust_file="$allreps_dir/cells.top${nclust_top}clust.csv"
    
head -n 1 "$cells_file" | cut -d ',' -f 1,4,5 > "$cells_topclust_file"
awk -F',' 'BEGIN{OFS=","}{ if ($4 ~ /'"$clusters_top_re"'/){ print $1, $4, $5 } }' "$cells_file"  >> "$cells_topclust_file"

# Count number of cells matching one of the chosen clusters
echo "Total cells across all chosen clusters (per "$(basename "$cells_topclust_file")"): "$( cut -d ',' -f 3 "$cells_topclust_file" | uniq | grep -v '^#' | wc -l )

                                                                                            
# ==== Label DNA MERFISH data from cells belonging to 'top' clusters (clusters with >= cutoff cells); discard data for other cells
dna_allreps_file="$allreps_dir/dna_merfish.top${nclust_top}clust.csv"

# Make header for dna_allreps_file
paste -d ',' \
    <( echo "#cell_id" ) \
    <( head -n 1 $(find "$sample_dir" -type f -path '*/biorep*_techrep*/dna_merfish.csv' | head -n 1 ) | sed 's/#cell_id,//' ) \
    <( head -n 1 "$cells_topclust_file" | cut -d ',' -f 2    ) > "$dna_allreps_file"

# Assign cluster label to DNA MERFISH data (for cells in the chosen clusters)
for biorep in $( find "$sample_dir" -type d -name 'biorep*_techrep*' ); do
    rep_id=$(basename "$biorep" | sed 's/_/./;s/[A-z]*//g' )
    awk -F ',' -v rep_id="$rep_id" '{ if (($1 == rep_id) || ($1 ~ /rep_id/)){ print $2 "," $3 } }' "$cells_topclust_file" | sort -t ',' -k 2b,2 | \
        join -t ',' -j 2 --header --check-order "$biorep/dna_merfish.csv" - | grep -v '^#' >> "$dna_allreps_file"
done

# Count number of cells matching one of the chosen clusters
echo "Total cells across all chosen clusters (per "$(basename "$dna_allreps_file")"): "$( cut -d ',' -f 1 "$dna_allreps_file" | uniq | grep -v '^#' | wc -l )

                                                                                      
# # ==== Segregate DNA MERFISH data by cluster

# Make directories per cluster, write header for sorted dna_merfish file
for cluster in $clusters_top; do
    mkdir -p "$allreps_dir/cluster/$cluster"
    head -n 1 "$dna_allreps_file" | cut -d ',' -f 1-10 > "$allreps_dir/cluster/$cluster/dna_merfish.$cluster.csv"
done

# Segregate DNA MERFISH data
awk -F',' -v outdir="$allreps_dir" 'BEGIN{OFS=","}{ if ($1 !~ /^#/) \
    { print $1, $2, $3, $4, $5, $6, $7, $8, $9, $10 >> outdir"/cluster/"$11"/dna_merfish."$11".csv" } }' "$dna_allreps_file"

# Count number of cells matching each of the chosen clusters
echo -e "\nCells per cluster:"
ncells_total=0
for cluster in $clusters_top; do
    ncells=$( cut -d ',' -f 1 "$allreps_dir/cluster/$cluster/dna_merfish.$cluster.csv" | uniq | grep -v '^#' | wc -l )
    echo "  $cluster: $ncells cells"
    ncells_total=$(($ncells_total + $ncells)) 
done
echo "TOTAL: $ncells_total cells"

Total cells across all chosen clusters (per cells.top8clust.csv): 33361
Total cells across all chosen clusters (per dna_merfish.top8clust.csv): 33361

Cells per cluster:
oligo: 6469 cells
l6_ct: 5463 cells
astro: 4830 cells
l2-3_it: 4373 cells
endo: 3687 cells
l4-5_it: 3370 cells
l5_it: 2659 cells
l6_it: 2510 cells
TOTAL: 33361 cells


## Characterizing the data

In [ ]:
%%bash

sample_dir="$HOME/noble_lab/repos/Chromatin_Analysis_MOp/data/MOp_wt"

allreps_dir="$sample_dir/all_reps"
clusters_dir="$allreps_dir/cluster"

cluster="astro"
cluster_file="$clusters_dir/$cluster/dna_merfish.$cluster.csv"

# All columns: 1=cell_id,  2=rep_id,  3=chrom,  4=trace_id,  5=chrom_order,  6=chrom_start,  7=chrom_end,  8=x,  9=y,  10=z

echo -e "\n\n==== QUESTION: Can a locus on a given 'chrom' be uniquely identified by either 'chrom_order', 'chrom_start', or 'chrom_end'? Or do you need some combo thereof?"

echo -e "\nInstances where a locus has the same start location but different end location (expected result=0):"
# Cols: #chrom,#chrom_start,#chrom_end
cut -d ',' -f 3,6,7 "$cluster_file" | sort | uniq | cut -d ',' -f 1-2 | uniq -d | wc -l

echo -e "\nInstances where a locus has the same start/end location but different value for 'chrom_order' (expected result=0):"
# Cols: #chrom,#chrom_start,#chrom_end
cut -d ',' -f 3,5,6,7 "$cluster_file" | sort | uniq | cut -d ',' -f 1-2 | uniq -d | wc -l

echo -e "\n\n==== CHARACTERIZING THE DATA"

echo -e "\nNumber of cells with data for each locus (on 1+ traces):"
# Cols: #cell_id,#chrom,#chrom_start,#chrom_end
cut -d ',' -f 1,3,6,7 "$cluster_file" | sort | uniq | cut -d ',' -f 1 | uniq -c | awk '!/#/ {print $1}' | sort -k1,1nr | pymath

echo -e "\nNumber of loci associated with each traceID:\nnum_loci\ttraceID"
# Cols: #chrom,#trace_id,#chrom_start,#chrom_end
cut -d ',' -f 3,4,6,7 "$cluster_file" | sort | uniq | cut -d ',' -f 2 | sort | uniq -c | awk '!/#/ {print $1 "\t\t" $2}' | sort -k1,1nr

echo -e "\nNumber of cells associated with each traceID:\nnum_cells\ttraceID"
# Cols: #cell_id,#trace_id
cut -d ',' -f 1,4 "$cluster_file" | sort | uniq | cut -d ',' -f 2 | sort | uniq -c | awk '!/#/ {print $1 "\t\t" $2}' | sort -k1,1nr

echo -e "\n\n==== QUESTION: Are there ever cases where a single copy of a single homolog of a single chromosome is represented by 2+ traces?"
#echo "If (A) and (B) below are identical, then the answer is 'no', otherwise its a 'maybe'..."
echo "Gotta be honest, the below tests don't really answer this question"

echo -e "\n(A) Number of cell-locus pairs associated with each traceID:\nnum_cell-locus_pairs\ttraceID"
# Cols: #cell_id,#chrom,#trace_id,#chrom_start,#chrom_end
# 1. Get unique cell-locus-traces
# 2. Number of cell-locus pairs per trace_ID
cut -d ',' -f 1,3,4,6,7 "$cluster_file" | sort | uniq | \
    cut -d ',' -f 3 | sort | uniq -c | awk '!/#/ {print $1 "\t\t\t" $2}' | sort -k1,1nr 

echo -e "\n(B) Number copies of each cell-locus pair:\nnum_cell-locus_pairs\tnum_copies_of_given_pair"
# # Cols: #cell_id,#chrom,#trace_id,#chrom_start,#chrom_end
# 1. Get unique cell-locus-traces
# 2. Number of unique traces per cell-locus pair
# 3. How often we see the given number of unique traces per cell-locus pair
cut -d ',' -f 1,3,4,6,7 "$cluster_file" | sort | uniq | \
    cut -d ',' -f 1,2,4,5 | sort | uniq -c | \
    awk '!/#/ {print $1}' | sort | uniq -c | awk '!/#/ {print $1 "\t\t\t" $2}' | sort -k1,1nr  # TODO is this correct?

# Filter and label DNA-MERFISH data

## Choose loci 2.5Mb apart

In [143]:
import os
import pandas as pd
import numpy as np
from process_loci import get_evenly_spaced_loci

sample_dir = "~/noble_lab/repos/Chromatin_Analysis_MOp/data/MOp_wt"
all_loci_file = os.path.join(sample_dir, 'supplementary/target_regions.bed.gz')
loci = get_evenly_spaced_loci(all_loci_file, spacing=2.5, cutoff_neighbor=0.05, cutoff_median=0.05, outdir=sample_dir, verbose=True)
loci

1981 loci... gaps between loci:
count    1961.000000
mean        1.289912
std         1.005944
min        -0.006165
25%         0.433977
50%         1.076219
75%         2.180035
max        12.502445

For loci whose midpoints differ by <0.05, choose locus with smallest absolute deviation from median offset
1940 loci... gaps between loci:
count    1920.000000
mean        1.317457
std         0.999483
min         0.050709
25%         0.468003
50%         1.097869
75%         2.217746
max        12.502445

Remove loci whose absolute deviation from median offset >= 0.05
987 loci... gaps between loci:
count    967.000000
mean       2.608568
std        0.664271
min        2.491560
25%        2.498452
50%        2.500004
75%        2.501743
max       12.502445

32 loci are spaced >4 Mb apart

Saving to: ~/noble_lab/repos/Chromatin_Analysis_MOp/data/MOp_wt/loci.spaced2500kb.csv


,chrom,start_bp,end_bp,locus_size,mid,offset,offset_vs_med,gap_from_prev,gap_to_next,idx_chrom,idx_genome
0,1,3742742,3759944,0.017202,3.751343,1.251343,-0.003087,NaN,2.501121,0,0
1,1,6245958,6258969,0.013011,6.252464,1.252464,-0.001966,2.501121,2.497498,1,1
2,1,8740008,8759916,0.019908,8.749962,1.249962,-0.004468,2.497498,2.502718,2,2
5,1,11247744,11257616,0.009872,11.252680,1.252680,-0.001750,2.502718,2.497225,3,3
6,1,13741888,13757922,0.016034,13.749905,1.249905,-0.004525,2.497225,2.505740,4,4
...,...,...,...,...,...,...,...,...,...,...,...
1933,X,158749404,158759978,0.010574,158.754691,1.254691,0.000262,2.504721,2.498749,61,210
1934,X,161247067,161259813,0.012746,161.253440,1.253440,-0.000989,2.498749,2.501180,62,211
1935,X,163750534,163758706,0.008172,163.754620,1.254620,0.000190,2.501180,2.499187,63,212
1936,X,166247682,166259932,0.012250,166.253807,1.253807,-0.000623,2.499187,2.498010,64,213


In [149]:
spacing = 2.5
loci_keep = get_evenly_spaced_loci(
    all_loci_file, spacing=spacing, cutoff_neighbor=spacing / 10,
    cutoff_median=spacing / 2, verbose=True)

assert (~loci.set_index(['chrom', 'start_bp', 'end_bp']).index.isin(loci_keep.set_index(['chrom', 'start_bp', 'end_bp']).index)).sum() == 0

loci_keep

1981 loci... gaps between loci:
count    1961.000000
mean        1.289912
std         1.005944
min        -0.006165
25%         0.433977
50%         1.076219
75%         2.180035
max        12.502445

For loci whose midpoints differ by <0.25, choose locus with smallest absolute deviation from median offset
1712 loci... gaps between loci:
count    1692.000000
mean        1.494883
std         0.966193
min         0.250058
25%         0.711069
50%         1.344810
75%         2.496650
max        12.502445

Remove loci whose absolute deviation from median offset >= 1.25
1710 loci... gaps between loci:
count    1690.000000
mean        1.496652
std         0.967308
min         0.250058
25%         0.710676
50%         1.351214
75%         2.496720
max        12.502445

21 loci are spaced >4 Mb apart


,chrom,start_bp,end_bp,locus_size,mid,offset,offset_vs_med,gap_from_prev,gap_to_next,idx_chrom,idx_genome
0,1,3742742,3759944,0.017202,3.751343,1.251343,-0.003035,NaN,2.501121,0,0
1,1,6245958,6258969,0.013011,6.252464,1.252464,-0.001915,2.501121,2.497498,1,1
2,1,8740008,8759916,0.019908,8.749962,1.249962,-0.004416,2.497498,0.882938,2,2
3,1,9627926,9637875,0.009949,9.632900,2.132900,0.878522,0.882938,1.619779,2,2
4,1,11247744,11257616,0.009872,11.252680,1.252680,-0.001698,1.619779,2.497225,3,3
...,...,...,...,...,...,...,...,...,...,...,...
1707,X,163750534,163758706,0.008172,163.754620,1.254620,0.000242,2.501180,2.499187,63,212
1708,X,166247682,166259932,0.012250,166.253807,1.253807,-0.000571,2.499187,0.908501,64,213
1709,X,167157164,167167452,0.010288,167.162308,2.162308,0.907930,0.908501,1.589509,64,213
1710,X,168746045,168757590,0.011545,168.751817,1.251817,-0.002561,1.589509,1.232429,65,214


## Exploring the DNA-MERFISH dataset

In [438]:
import os
import pandas as pd
import numpy as np

sample_dir = "~/noble_lab/repos/Chromatin_Analysis_MOp/data/MOp_wt"
clusters_dir = os.path.join(sample_dir, "all_reps", "cluster")

cluster = "astro"
cluster_file = os.path.join(clusters_dir, f"{cluster}/dna_merfish.{cluster}.csv")

In [439]:
df = pd.read_csv(cluster_file, dtype={
    'cell_id': int, 'rep_id': float, 'chrom': str, 'trace_id': int, 'chrom_order': int,
    'chrom_start': int, 'chrom_end': int, 'x': float, 'y': float, 'z': float})
df.columns = [x.strip('#') for x in df.columns]

if 'hmlg' in df.columns:  # TODO temp fix
    df.rename({'hmlg': 'trace_id'}, axis=1, inplace=True)

### Questions about traces

In [ ]:
def characterize_trace(df):
    s = pd.Series({
        'trace_start': df.chrom_order.min(), 'trace_end': df.chrom_order.max(),
        'trace_len': len(df), 'trace_nloci': df.chrom_order.max() + 1 - df.chrom_order.min(),
        'trace_nbp': df.chrom_end.max() - df.chrom_start.min()})
    return s
    

df_trace = df.groupby(['cell_id', 'chrom', 'trace_id']).apply(characterize_trace, include_groups=False).reset_index()
# df_trace.rename({0: 'nloci'}, axis=1, inplace=True)

#### 1. Does the trace_id correspond to the relative length of the trace, such that trace_id=1 is always longer than trace_id=2, etc? (→ NO!)

In [39]:
def compare_trace_for_chrom(df, compare_via='trace_len'):
    if df.trace_id.max() == 1:
        return None
    expected_trace_order = np.arange(1, df.trace_id.max() + 1)
    actual_trace_order = df.sort_values(compare_via, ascending=False).trace_id.values
    if np.array_equal(actual_trace_order, expected_trace_order):
        return None
    return df[expected_trace_order != actual_trace_order].reset_index(drop=True)


for compare_via in ('trace_len', 'trace_nloci', 'trace_nbp'):
    res = df_trace.groupby(['cell_id', 'chrom']).apply(compare_trace_for_chrom, include_groups=False, compare_via=compare_via).reset_index(
        level=[0, 1]).set_index(['cell_id', 'chrom'])
    print(f"{compare_via=}, num cell-chrom combos where lower-numbered traces don't have the most loci={len(res.index.drop_duplicates())}")
    print(res.trace_id.drop_duplicates().values)

# num_suprising_ordering = res.groupby(['cell_id', 'chrom']).size().reset_index()
# len(num_suprising_ordering.loc[num_suprising_ordering[0] != 2, 'cell_id'].drop_duplicates())

compare_via='trace_len', num cell-chrom combos where lower-numbered traces don't have the most loci=2169
[1 2 3 4 5]
compare_via='trace_nloci', num cell-chrom combos where lower-numbered traces don't have the most loci=2144
[1 2 3 4]
compare_via='trace_nbp', num cell-chrom combos where lower-numbered traces don't have the most loci=2311
[1 2 3 4]


#### 2. Are there ever totally non-overlapping (complementary) traces? (→ NO!)

In [80]:
def find_nonoverlapping_traces(df):
    if len(df) == 1:
        return None
    traces = df.trace_id.values
    df_overlap = []
    for i in traces:
        start_i, end_i = df.loc[df.trace_id == i, ['trace_start', 'trace_end']].values.ravel()
        for j in range(i + 1, traces.max()):
            start_j, end_j = df.loc[df.trace_id == j, ['trace_start', 'trace_end']].values.ravel()
            if (end_j < start_i) or (end_i < start_j):  # Traces don't overlap
                df_overlap.append({
                    'trace_id': [i, j][np.argmin([start_i, start_j])],
                    'trace_start': np.min([start_i, start_j]),
                    'trace_max': np.max([end_i, end_j])})
    if not len(df_overlap):
        return None
    return pd.Dataframe(df_overlap)


has_overlap = df_trace.groupby(['cell_id', 'chrom']).apply(find_nonoverlapping_traces, include_groups=False)
if not len(has_overlap):
    print("No non-overlapping traces!")
has_overlap

No non-overlapping traces!


""


#### 3. Further characterize traces

In [91]:
df_chrom_ntraces = df_trace.groupby(['cell_id', 'chrom']).size().reset_index().rename({0: 'ntraces'}, axis=1)
df_chrom_ntraces

,cell_id,chrom,ntraces
0,100248997715285942293181957009945158269,12,1
1,100248997715285942293181957009945158269,4,1
2,100267022990578563888527231462820232415,17,1
3,100267022990578563888527231462820232415,18,1
4,100267022990578563888527231462820232415,2,1
...,...,...,...
30448,99885450568263402858676742529872331188,18,1
30449,99885450568263402858676742529872331188,19,1
30450,99885450568263402858676742529872331188,2,1
30451,99885450568263402858676742529872331188,5,1


In [119]:
# for ntraces in df_cell_desc_chrom.ntraces.drop_duplicates().values
ntraces_types = {
    '1 trace': df_chrom_ntraces.ntraces == 1,
    '2 traces': df_chrom_ntraces.ntraces == 2,
    '>2 traces': df_chrom_ntraces.ntraces > 2}

df_chrom_ntraces[df_chrom_ntraces.ntraces > 2].groupby('cell_id').size().value_counts().sum()

df_chrom_ntraces[df_chrom_ntraces.ntraces == 2].groupby('cell_id').size().value_counts()

1     564
2     290
3     208
4     138
5     108
6      99
8      61
7      60
9      52
10     39
11     29
13     21
14     18
12     16
16     10
15     10
17      8
18      5
Name: count, dtype: int64

In [90]:
df_cell_desc_gt2trace = df_chrom_ntraces[df_chrom_ntraces.ntraces > 2].groupby('cell_id').size().reset_index().rename({0: 'nchrom_gt2trace'}, axis=1).sort_values(
    'nchrom_gt2trace', ascending=False)
print(df_cell_desc_gt2trace.nchrom_gt2trace.sum())
print(len(df_cell_desc_gt2trace))
print((df_cell_desc_gt2trace.nchrom_gt2trace > 1).sum())
df_cell_desc_gt2trace

200
99
34


,cell_id,nchrom_gt2trace
49,236707373299231246192523599949109740194,15
92,796907248109666331037430947342595009,7
42,211682870950575149725950043686209170640,7
87,57089633890354238603235644175840384596,7
57,266648537481793942323173786920475723304,7
...,...,...
36,202495501063792655693561618949921176863,1
35,201213478246125381219227212572258749785,1
34,201069040844675365475257195673380437661,1
32,197940930151083433626300695622139959148,1


In [86]:
df_cell_desc_2trace = df_chrom_ntraces[df_chrom_ntraces.ntraces == 2].groupby('cell_id').size().reset_index().rename({0: 'nchrom_2trace'}, axis=1).sort_values(
    'nchrom_2trace', ascending=False)

print(df_cell_desc_2trace.nchrom_2trace.max())
print(len(df_cell_desc_2trace))
print(len(df_cell_desc_2trace[df_cell_desc_2trace.nchrom_2trace >= 2]))
print(len(df_cell_desc_2trace[df_cell_desc_2trace.nchrom_2trace >= 10]))
print(len(df_cell_desc_2trace[df_cell_desc_2trace.nchrom_2trace >= 15]))
print(len(df_cell_desc_2trace[df_cell_desc_2trace.nchrom_2trace >= 18]))

18
1736
1172
156
33
5


In [89]:
df_chrom_2trace = df_chrom_ntraces[df_chrom_ntraces.ntraces == 2].groupby('chrom').size().reset_index().rename({0: 'ncell_2trace'}, axis=1).sort_values(
    'ncell_2trace', ascending=False)

df_chrom_2trace

,chrom,ncell_2trace
1,10,497
12,3,476
17,8,425
13,4,420
14,5,413
9,18,397
3,12,368
4,13,367
5,14,357
15,6,349


### Question: are any of the imaged loci missing in all cells? (→NO!)

In [150]:
def find_missing_loci(df):
    if len(df) == df.chrom_order.max() + 1 - df.chrom_order.min():
        return False
    return True

chrom_has_missing_loci = df[['chrom', 'chrom_order']].drop_duplicates().groupby('chrom').apply(find_missing_loci, include_groups=False)
chrom_has_missing_loci.sum()

0

### Question: do some cells really have >2 copies of the same exact locus? (→YES!)

In [92]:
def count_occurances_of_locus(df, n_traces):
    s = pd.Series({'ncopies': len(df), 'ncells': len(df.cell_id.drop_duplicates())})
    for i in range(1, n_traces + 1):
        s[f'ncells_{i}copies'] = (df.groupby('cell_id').size() == i).sum()
    s['ncells_min2copies'] = (df.groupby('cell_id').size() >= 2).sum()
    return s

def add_missing_loci(df):
    if len(df) == df.chrom_order.max() + 1 - df.chrom_order.min():
        return df
    bins_all = np.arange(df.chrom_order.min(), df.chrom_order.max() + 1, dtype=int)
    is_missing = np.isin(bins_all, df.chrom_order.values, assume_unique=True, invert=True)

    df_missing = pd.DataFrame()
    df_missing['chrom_order'] = bins_all[is_missing]
    df_full = pd.concat([df, df_missing]).sort_values('chrom_order').fillna(0).reset_index(drop=True)
    for col in df.columns:
        df_full[col] = df_full[col].astype(int)
    return df_full


# per_locus = df.groupby(['chrom', 'chrom_order']).size().reset_index()
# per_locus.rename({0: 'ncopies'}, axis=1, inplace=True)
# # per_locus = per_locus.drop(index=[1, 3]).reset_index(drop=True) # FIXME remove, just for testing
# per_locus = per_locus.groupby('chrom').apply(add_missing_loci, include_groups=False).reset_index(level=0)
# per_locus['ncopies'].describe()

n_traces = df.trace_id.max()
per_locus = df.groupby(['chrom', 'chrom_order']).apply(count_occurances_of_locus, include_groups=False, n_traces=n_traces).reset_index()
for i in range(1, n_traces + 1):
    print(i, per_locus[f'ncells_{i}copies'].sum())
    if per_locus[f'ncells_{i}copies'].sum() == 0:
        per_locus.drop(f'ncells_{i}copies', axis=1)
per_locus = per_locus.groupby('chrom').apply(add_missing_loci, include_groups=False).reset_index(level=0)
print(per_locus['ncopies'].describe())
per_locus

1 1465881
2 178722
3 2863
4 188
5 18
count    1981.000000
mean      925.167087
std       222.108784
min        64.000000
25%       805.000000
50%       932.000000
75%      1065.000000
max      1896.000000
Name: ncopies, dtype: float64


,chrom,chrom_order,ncopies,ncells,ncells_1copies,ncells_2copies,ncells_3copies,ncells_4copies,ncells_5copies,ncells_min2copies
0,1,0,1133,1001,872,126,3,0,0,129
1,1,1,840,758,677,80,1,0,0,81
2,1,2,616,577,538,39,0,0,0,39
3,1,3,956,853,751,101,1,0,0,102
4,1,4,791,733,675,58,0,0,0,58
...,...,...,...,...,...,...,...,...,...,...
1976,X,59,388,388,388,0,0,0,0,0
1977,X,60,423,420,417,3,0,0,0,3
1978,X,61,416,413,410,3,0,0,0,3
1979,X,62,401,400,399,1,0,0,0,1


## Filtering DNA-MERFISH data

### Submit to cluster

In [ ]:
%%bash

sample_dir="$HOME/noble_lab/repos/Chromatin_Analysis_MOp/data/MOp_wt"
chosen_loci_file="$sample_dir/loci.spaced2500kb.csv"
allreps_dir="$sample_dir/all_reps"

ntraces_per_cell='all'
interpolate=false # keep false
compare_to_minimal_cells=false # keep false
inverse_disterror=false # keep false
compare_to_2traces_per_cell=true  # keep true
compared_loci_cutoff=''
nrmse_method='' 
root_mse=false
hmlg_err_ratio_cutoff=0

# desc='.r0'


declare -a SUBDIR=()
declare -a EXTRA_ARGS=()
if [[ "$hmlg_err_ratio_cutoff" == "0" ]]; then
    SUBDIR+=("compare_to_1cell")
    EXTRA_ARGS+=("--hmlg_err_ratio_cutoff 0")
fi
if [[ "$ntraces_per_cell" == 1 ]] || [[ "$ntraces_per_cell" == 2 ]]; then
    SUBDIR+=("ntraces_chrom-cell_$ntraces_per_cell")
    EXTRA_ARGS+=("--ntraces_per_cell $ntraces_per_cell")
elif $compare_to_2traces_per_cell && [[ "$hmlg_err_ratio_cutoff" != "0" ]]; then
    SUBDIR+=("compare_to_2traces_per_chrom-cell")
    EXTRA_ARGS+=("--compare_to_2traces_per_cell")
fi
if $compare_to_minimal_cells; then
    SUBDIR+=("compare_to_minimal_cells")
    EXTRA_ARGS+=("--compare_to_minimal_cells")
fi
if [[ "$compared_loci_cutoff" != '' ]] && [[ "$compared_loci_cutoff" != "0" ]]; then
    cutoff_perc=$( printf "%g" $( bc -l <<< "$compared_loci_cutoff*100" ) )
    SUBDIR+=("compared_loci_${cutoff_perc}p")
    EXTRA_ARGS+=("--compared_loci_cutoff $compared_loci_cutoff")
fi
if $interpolate; then
    SUBDIR+=("interp")
    EXTRA_ARGS+=("--interpolate")
fi
if [[ "$nrmse_method" == 'none' ]] || [[ "$nrmse_method" == 'None' ]]; then
    nrmse_method=''
fi
if $inverse_disterror || [[ "$nrmse_method" != '' ]] || $root_mse; then
    declare -a TMP=("disterror")
    if $inverse_disterror; then
        TMP+=("inverse")
        EXTRA_ARGS+=("--inverse_disterror")
    fi
    if [[ "$nrmse_method" != '' ]]; then
        TMP+=("nrmse-$nrmse_method")
        EXTRA_ARGS+=("--nrmse_method $nrmse_method")
    elif $root_mse; then
        TMP+=("rmse")
        EXTRA_ARGS+=("--root_mse")
    fi
    TMP=$(IFS=_ ; echo "${TMP[*]}")
    SUBDIR+=("$TMP")
fi
if [[ "$hmlg_err_ratio_cutoff" != '' ]] && [[ "$hmlg_err_ratio_cutoff" != "1" ]] && [[ "$hmlg_err_ratio_cutoff" != "0" ]]; then
    cutoff_perc=$( printf "%g" $( bc -l <<< "$hmlg_err_ratio_cutoff*100" ) )
    SUBDIR+=("hmlg_err_ratio_${cutoff_perc}p")
    EXTRA_ARGS+=("--hmlg_err_ratio_cutoff $hmlg_err_ratio_cutoff")
fi
SUBDIR="/"$(IFS=. ; echo "${SUBDIR[*]}")
EXTRA_ARGS=$(IFS=' ' ; echo "${EXTRA_ARGS[*]}")

echo -e "SUBDIR = $SUBDIR\n\n"

sge_outdir="$allreps_dir/sge_out/filter.hmlg$SUBDIR"
mkdir -p "$sge_outdir"
for cluster in $( ls "$allreps_dir/cluster" ); do
    merfish_file="$allreps_dir/cluster/$cluster/dna_merfish.$cluster.csv"
    final_outfile="$allreps_dir/cluster/$cluster$SUBDIR/dna_merfish.$cluster.filter.hmlg.csv"
    if [ ! -f "$final_outfile" ]; then
        echo -e "\n\n==== CLUSTER=$cluster"
        # mv "$allreps_dir/cluster/$cluster/per_chrom" "$allreps_dir/cluster/$cluster/per_chrom.old"
        for chrom in $(seq 1 19); do
            outfile="$allreps_dir/cluster/$cluster$SUBDIR/per_chrom/dna_merfish.$cluster.filter.hmlg.chrom_$chrom.csv"
            if [ ! -f "$outfile" ]; then
                echo -e "\nCLUSTER=$cluster, CHROM=$chrom"    
                qsub -P sage -cwd -V -l h_rt=7:00:00,mfree=3G -o "$sge_outdir" -j "yes" -N "hmlg.$cluster.chr$chrom$desc" -R y -b y \
                python parse_dna_merfish.py "$merfish_file" --chosen_loci "$chosen_loci_file" $EXTRA_ARGS \
                --chrom "$chrom" --verbose
            fi
        done
    fi
done

In [ ]:
%%bash

gcq stat x | grep 'hmlg\.' | sed -r 's/.*\.chr[0-9][0-9]*\.*//;s/ (0|-1)$//' | uniq -c | awk '{print $1 "\t" $2}' | sort -k2,2

### Consolidate filtered data across all chromosomes

In [6]:
%%bash

sample_dir="$HOME/noble_lab/repos/Chromatin_Analysis_MOp/data/MOp_wt"
chosen_loci_file="$sample_dir/loci.spaced2500kb.csv"
allreps_dir="$sample_dir/all_reps"


sge_outdir="$allreps_dir/sge_out"
mkdir -p "$sge_outdir"

ntraces_per_cell='all'
interpolate=false # keep false
compare_to_minimal_cells=false # keep false
inverse_disterror=false # keep false
compare_to_2traces_per_cell=true  # keep true
compared_loci_cutoff=''
nrmse_method='' 
root_mse=false
hmlg_err_ratio_cutoff=0


declare -a SUBDIR=()
if [[ "$hmlg_err_ratio_cutoff" == "0" ]]; then
    SUBDIR+=("compare_to_1cell")
fi
if [[ "$ntraces_per_cell" == 1 ]] || [[ "$ntraces_per_cell" == 2 ]]; then
    SUBDIR+=("ntraces_chrom-cell_$ntraces_per_cell")
elif $compare_to_2traces_per_cell && [[ "$hmlg_err_ratio_cutoff" != "0" ]]; then
    SUBDIR+=("compare_to_2traces_per_chrom-cell")
fi
if $compare_to_minimal_cells; then
    SUBDIR+=("compare_to_minimal_cells")
fi
if [[ "$compared_loci_cutoff" != '' ]] && [[ "$compared_loci_cutoff" != "0" ]]; then
    cutoff_perc=$( printf "%g" $( bc -l <<< "$compared_loci_cutoff*100" ) )
    SUBDIR+=("compared_loci_${cutoff_perc}p")
fi
if $interpolate; then
    SUBDIR+=("interp")
fi
if [[ "$nrmse_method" == 'none' ]] || [[ "$nrmse_method" == 'None' ]]; then
    nrmse_method=''
fi
if $inverse_disterror || [[ "$nrmse_method" != '' ]] || $root_mse; then
    declare -a TMP=("disterror")
    if $inverse_disterror; then
        TMP+=("inverse")
    fi
    if [[ "$nrmse_method" != '' ]]; then
        TMP+=("nrmse-$nrmse_method")
    elif $root_mse; then
        TMP+=("rmse")
    fi
    TMP=$(IFS=_ ; echo "${TMP[*]}")
    SUBDIR+=("$TMP")
fi
if [[ "$hmlg_err_ratio_cutoff" != '' ]] && [[ "$hmlg_err_ratio_cutoff" != "1" ]] && [[ "$hmlg_err_ratio_cutoff" != "0" ]]; then
    cutoff_perc=$( printf "%g" $( bc -l <<< "$hmlg_err_ratio_cutoff*100" ) )
    SUBDIR+=("hmlg_err_ratio_${cutoff_perc}p")
fi
SUBDIR="/"$(IFS=. ; echo "${SUBDIR[*]}")

# Check that all data is available
proceed=true
for cluster in $(ls "$allreps_dir/cluster"); do
    # echo -e "\n==== CLUSTER=$cluster"
    outfile="$allreps_dir/cluster/$cluster$SUBDIR/dna_merfish.$cluster.filter.hmlg.csv"
    if [ ! -f "$outfile" ]; then
        missing_chrom=""
        for chrom in $(seq 1 19); do
            merfish_file="$allreps_dir/cluster/$cluster$SUBDIR/per_chrom/dna_merfish.$cluster.filter.hmlg.chrom_$chrom.csv" 
            # echo "$allreps_dir/cluster/$cluster$SUBDIR/per_chrom"
            if [ ! -f "$merfish_file" ]; then
                # echo "ERROR: data not found for $cluster, chr$chrom" #>&2
                proceed=false
                missing_chrom="$missing_chrom$chrom, "
            fi
        done
        if [[ "$missing_chrom" == "" ]]; then
            echo -e "$cluster\t... success" #: "$( echo "$allreps_dir/cluster/$cluster$SUBDIR/per_chrom" | sed 's|^'$HOME'|~|')
            # Write header
            head -n 1 "$allreps_dir/cluster/$cluster$SUBDIR/per_chrom/dna_merfish.$cluster.filter.hmlg.chrom_1.csv"  > "$outfile"
            # Consolidate data across chromosomes
            grep -r "$allreps_dir/cluster/$cluster$SUBDIR/per_chrom/dna_merfish.$cluster.filter.hmlg.chrom_"*.csv -hve '^cell_id' | \
                sort -t ',' -k1,1n -k12,12n -k2,2n -k3,3n >> "$outfile" && rm -rf "$allreps_dir/cluster/$cluster$SUBDIR/per_chrom"
        else
            echo -e "$cluster\t... missing chromosomes: $missing_chrom" #>&2
        fi
    fi
done
# if ! $proceed; then
#     echo "ERROR: missing data for one or more clusters/chromosomes" >&2
#     exit 1
# fi

# Prepare dataset: DNA-MERFISH & snm3c-seq

## Save distances

In [ ]:
%%bash

sample_dir="$HOME/noble_lab/repos/Chromatin_Analysis_MOp/data/MOp_wt"
chosen_loci_file="$sample_dir/loci.spaced2500kb.csv"
allreps_dir="$sample_dir/all_reps"

min_nonmissing=0.05
nmol_per_hmlg_ratio=1


interpolate=false; inverse_disterror=false; compare_to_minimal_cells=false # keep false
compared_loci_cutoff=''; nrmse_method=''; root_mse=false
compare_to_2traces_per_cell=false
# ntraces_per_cell=2; hmlg_err_ratio_cutoff=1
ntraces_per_cell='all'; hmlg_err_ratio_cutoff=0


declare -a SUBDIR=()
if [[ "$hmlg_err_ratio_cutoff" == "0" ]]; then
    SUBDIR+=("compare_to_1cell")
fi
if [[ "$ntraces_per_cell" == 1 ]] || [[ "$ntraces_per_cell" == 2 ]]; then
    SUBDIR+=("ntraces_chrom-cell_$ntraces_per_cell")
elif $compare_to_2traces_per_cell && [[ "$hmlg_err_ratio_cutoff" != "0" ]]; then
    SUBDIR+=("compare_to_2traces_per_chrom-cell")
fi
if $compare_to_minimal_cells; then
    SUBDIR+=("compare_to_minimal_cells")
fi
if [[ "$compared_loci_cutoff" != '' ]] && [[ "$compared_loci_cutoff" != "0" ]]; then
    cutoff_perc=$( printf "%g" $( bc -l <<< "$compared_loci_cutoff*100" ) )
    SUBDIR+=("compared_loci_${cutoff_perc}p")
fi
if $interpolate; then
    SUBDIR+=("interp")
fi
if [[ "$nrmse_method" == 'none' ]] || [[ "$nrmse_method" == 'None' ]]; then
    nrmse_method=''
fi
if $inverse_disterror || [[ "$nrmse_method" != '' ]] || $root_mse; then
    declare -a TMP=("disterror")
    if $inverse_disterror; then
        TMP+=("inverse")
    fi
    if [[ "$nrmse_method" != '' ]]; then
        TMP+=("nrmse-$nrmse_method")
    elif $root_mse; then
        TMP+=("rmse")
    fi
    TMP=$(IFS=_ ; echo "${TMP[*]}")
    SUBDIR+=("$TMP")
fi
if [[ "$hmlg_err_ratio_cutoff" != '' ]] && [[ "$hmlg_err_ratio_cutoff" != "1" ]] && [[ "$hmlg_err_ratio_cutoff" != "0" ]]; then
    cutoff_perc=$( printf "%g" $( bc -l <<< "$hmlg_err_ratio_cutoff*100" ) )
    SUBDIR+=("hmlg_err_ratio_${cutoff_perc}p")
fi
SUBDIR="/"$(IFS=. ; echo "${SUBDIR[*]}")

echo "SUBDIR: $SUBDIR"

# # qlogin -l mfree=200G
# cd /net/gs/vol1/home/gesine/noble_lab/repos/Chromatin_Analysis_MOp/jupyter ####
# cluster=astro #endo #l5_it #astro ####
# merfish_file="$allreps_dir/cluster/$cluster$SUBDIR/dna_merfish.$cluster.filter.hmlg.csv" ####
# # counts_args="--k -0 --d0 0 --nu 0" # counts_args="--k -14.71465111 --d0 0.4515491" #"--alpha -2" ## counts_args="--contact_th 0.75"
# # counts_args="--k -5.16 --d0 2.69 --nu 1.23e-05"  # obj=-0.88644655 → PEARSON.infer_nu.kmax-4_buffer1e-5_d0min50p, seed=0  (FACTR)
# # counts_args="--k -9.12 --d0 1.72 --nu 1e-05"  #obj=0.18979348 → RMSE.infer_nu.kmax-4_buffer1e-5_d0min50p, seed=0  (FACTR)
# counts_args="--k -12.34 --d0 1.36 --nu 5.666e-06" # obj=0.004587645749   M=0.00016385  C=0.00044238 → mse.infer_nu.constraint10_w-1, seed=2  (FACTR) →→ exp=-0.605 (snm3c = -0.886)
# python coords_to_matrix.py "$merfish_file" --min_nonmissing_per_phased_locus "$min_nonmissing" $counts_args --nmol_per_hmlg_ratio "$nmol_per_hmlg_ratio" ####


# counts_args="--k -12.34 --d0 1.36 --nu 5.666e-06" # obj=0.004587645749   M=0.00016385  C=0.00044238 → mse.infer_nu.constraint10_w-1, seed=2  (FACTR) →→ exp=-0.605 (snm3c = -0.886)
counts_args="--k -12.34 --d0 1.36 --nu 5.666e-06 --scale_counts_by 1e20" # obj=0.004587645749   M=0.00016385  C=0.00044238 → mse.infer_nu.constraint10_w-1, seed=2  (FACTR) →→ exp=-0.605 (snm3c = -0.886)


sge_outdir="$allreps_dir/sge_out/coords_to_matrix/$SUBDIR"
mkdir -p "$sge_outdir"

for cluster in $( ls "$allreps_dir/cluster" | grep -E 'astro' ); do  # l2-3_it|endo  # l6_ct|oligo
    merfish_file="$allreps_dir/cluster/$cluster/$SUBDIR/dna_merfish.$cluster.filter.hmlg.csv"
    echo -e "\nCLUSTER=$cluster"
    qsub -P noblelab -cwd -V -l mfree=160G -o "$sge_outdir" -j "yes" -N "c2m.$cluster.logistic.1e20."$(echo "$SUBDIR" | sed 's|/||') -R y -b y \
    python coords_to_matrix.py "$merfish_file" --min_nonmissing_per_phased_locus "$min_nonmissing" $counts_args \
    --nmol_per_hmlg_ratio "$nmol_per_hmlg_ratio"
done

### Load data and plot manually

#### Load

In [2]:
import numpy as np
import pandas as pd
import os
import re
from iced.io import write_counts, write_lengths
from topsy.plot.plot_distances import plot_distance_matrix
from topsy.plot.plot_counts import plot_counts_single
from scipy import sparse
import seaborn as sns
import matplotlib.pyplot as plt
from coords_to_matrix import process_sc_dna_coords
from parse_dna_merfish import get_full_outdir
from process_counts import get_beta_ua, get_unambig_counts

nreads = 1e8
nmol_per_hmlg_ratio = 1

# ntraces_per_cell = 2; hmlg_err_ratio_cutoff = 1
ntraces_per_cell = None; hmlg_err_ratio_cutoff = 0


compare_to_2traces_per_cell = False; nrmse_method = None; root_mse = False; spacing = 2.5
sample_dir = "/net/gs/vol1/home/gesine/noble_lab/repos/Chromatin_Analysis_MOp/data/MOp_wt"
all_clusters_dir = os.path.join(sample_dir, "all_reps", "cluster")
cluster = "astro"
cluster_dir = os.path.join(all_clusters_dir, cluster)

# Get input_file
output_dir_tmp = get_full_outdir(
    cluster_dir, ntraces_per_cell=ntraces_per_cell, compare_to_labeled_loci_with_2_traces=False,
    compare_to_2traces_per_cell=compare_to_2traces_per_cell, compared_loci_cutoff=None,
    compare_to_minimal_cells=False, interpolate=False, inverse_disterror=False,
    nrmse_method=nrmse_method, root_mse=root_mse, hmlg_err_ratio_cutoff=hmlg_err_ratio_cutoff)
input_file = os.path.join(output_dir_tmp, f"dna_merfish.{cluster}.filter.hmlg.csv")

# Get outdir for counts
outdir_counts = os.path.join(
    sample_dir, "counts", f"unambig.nreads{nreads:.3g}".replace('e+0', 'e').replace('e+', 'e'), cluster)

#### Plot distances

In [8]:
def plot_distances_chrom(matrices, lengths_df, chrom, agg_type='mean'):
    lengths_s = lengths_df.groupby('chrom').size().sort_values(ascending=False)
    if isinstance(chrom, int) and chrom not in lengths_s.index.values:
        chrom = lengths_s.index.values[chrom]

    lengths_boundaries = pd.DataFrame(lengths_s).rename({0: 'length'}, axis=1)
    lengths_boundaries['begin'] = np.append(0, lengths_s.cumsum().values[:-1])
    lengths_boundaries['end'] = lengths_s.cumsum().values
    
    begin, end = lengths_boundaries.loc[chrom, ['begin', 'end']].values
    chrom_idx = np.append(np.arange(begin, end), np.arange(begin, end) + lengths_s.sum())
    data = matrices[f'dis_{agg_type}'][chrom_idx][:, chrom_idx]
    
    plot_distance_matrix(
        data, lengths=lengths_s[chrom],
        ploidy=2, title=f"Distances ({agg_type} across cells): {chrom}", outfile=None)

In [ ]:
lengths_s = lengths_df.groupby('chrom').size().sort_values(ascending=False)
for chrom in lengths_s.index:
    plot_distances_chrom(matrices, lengths_df=lengths_df, chrom=chrom, agg_type='mean')

## Save dataset

In [ ]:
%%bash

sample_dir="$HOME/noble_lab/repos/Chromatin_Analysis_MOp/data/MOp_wt"
chosen_loci_file="$sample_dir/loci.spaced2500kb.csv"
allreps_dir="$sample_dir/all_reps"
SUBDIR='/compare_to_1cell'

outdir="$HOME/noble_lab/projects/2015_gesine_diploid/data/liu2025"
sge_outdir="$allreps_dir/sge_out/make_dataset/$SUBDIR"

min_percentile_loci_cov='0.10'
infer_alpha_dis='mean'
num_infer_alpha=10
infer_alpha_mods="beta_from_intra_only"
ambiguity='ua'

cd /net/gs/vol1/home/gesine/noble_lab/repos/Chromatin_Analysis_MOp/jupyter
cluster=astro
merfish_file="$allreps_dir/cluster/$cluster$SUBDIR/dna_merfish.$cluster.filter.hmlg.csv"

python save_dataset.py "$merfish_file" --outdir "$outdir" --min_percentile_loci_cov "$min_percentile_loci_cov" --ambiguity "$ambiguity" \
--infer_alpha_mods $infer_alpha_mods --infer_alpha_dis "$infer_alpha_dis" --num_infer_alpha "$num_infer_alpha" --nreads '1e9'

# mkdir -p "$sge_outdir"
# for cluster in $( ls "$allreps_dir/cluster" | head -n 1 ); do 
#     merfish_file="$allreps_dir/cluster/$cluster/$SUBDIR/dna_merfish.$cluster.filter.hmlg.csv"
#     echo -e "\nCLUSTER=$cluster"
#     qsub -P sage -cwd -V -l mfree=10G -o "$sge_outdir" -j "yes" -N "data.$cluster" -R y -b y \
#     process_counts.py "$merfish_file" --outdir "$outdir" --min_percentile_loci_cov "$min_percentile_loci_cov" --ambiguity "$ambiguity" \
#     --infer_alpha_mods $infer_alpha_mods --infer_alpha_dis "$infer_alpha_dis" --num_infer_alpha "$num_infer_alpha"
# done